In [1]:
from langchain.tools import tool, ToolRuntime
from langchain.agents import AgentState

# Fix: Define the missing state schema for the email
class CustomState(AgentState):
    email: str

@tool
def read_email(runtime: ToolRuntime) -> str:
    """Read an email from the given address."""
    # Fix: Safely access the email from the state
    return runtime.state.get("email", "No email found in state.")

@tool
def send_email(body: str) -> str:
    """Send an email to the given address with the given subject and body."""
    # Fake email sending
    return "Email sent"

In [7]:
from langchain.agents import create_agent
from langchain_ollama import ChatOllama
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage
from langchain.agents.middleware import HumanInTheLoopMiddleware

model = ChatOllama(model="llama3.2:3b", temperature=0)

# Compilation with the HITL Middleware to target specific tools
agent = create_agent(
    model=model,
    tools=[read_email, send_email],
    state_schema=CustomState,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email": False,
                "send_email": True, # Halts execution ONLY before send_email
            },
            description_prefix="Tool execution requires approval",
        ),
    ],
)

config = {"configurable": {"thread_id": "hitl_session_1"}}

question = "Please read my email and send a polite reply confirming that I received it and will act accordingly."

# Invoke the agent and inject the fake email into the state
response = agent.invoke(
    {
        "messages": [HumanMessage(content=question)],
        "email": "Bonjour Sara, je vais être en retard pour notre réunion de demain. Pouvons-nous la reprogrammer? Cordialement, Sofia"
    },
    config=config
)

print("[GRAPH PAUSED] Waiting for human validation...")
print(response)

[GRAPH PAUSED] Waiting for human validation...
{'messages': [HumanMessage(content='Please read my email and send a polite reply confirming that I received it and will act accordingly.', additional_kwargs={}, response_metadata={}, id='fce65fcc-6402-4b0f-8cbc-37ff9925d3dd'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2:3b', 'created_at': '2026-06-16T13:02:47.201172Z', 'done': True, 'done_reason': 'stop', 'total_duration': 854748333, 'load_duration': 94796500, 'prompt_eval_count': 197, 'prompt_eval_duration': 60532833, 'eval_count': 68, 'eval_duration': 676168790, 'logprobs': None, 'model_name': 'llama3.2:3b', 'model_provider': 'ollama'}, id='lc_run--019ed086-f649-7772-91ac-0f9020fcf759-0', tool_calls=[{'name': 'read_email', 'args': {'address': "<user's email address>"}, 'id': '1059de46-40b8-45e4-a22a-0e605a37e414', 'type': 'tool_call'}, {'name': 'send_email', 'args': {'address': "<user's email address>", 'body': {'type': 'string', 'value': 'Thank you 

In [8]:
from langgraph.types import Command

response_approve = agent.invoke(
    Command(
        resume={"decisions": [{"type": "approve"}]}
    ),
    config=config
)

print("Approved Execution Result:")
print(response_approve['messages'][-1].content)

Approved Execution Result:
Bonjour Sara,

Je vais être en retard pour notre réunion de demain. Pouvons-nous la reprogrammer?

Cordialement,
Sofia


In [9]:

response_reject = agent.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "reject",
                    "message": "J'annule notre rendez-vous. Ne pas envoyer."
                }
            ]
        }
    ),
    config=config
)

print("Rejected Execution Result:")
print(response_reject['messages'][-1].content)


Rejected Execution Result:
Bonjour Sara,

Je vais être en retard pour notre réunion de demain. Pouvons-nous la reprogrammer?

Cordialement,
Sofia


In [10]:
response_edit = agent.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "edit",
                    "edited_action": {
                        "name": "send_email",
                        "args": {
                            "body": "Je suis désolée mais je ne serai pas disponible demain. Reprogrammons à la semaine prochaine."
                        }
                    }
                }
            ]
        }
    ),
    config=config
)

print("Edited Execution Result:")
print(response_edit['messages'][-1].content)

Edited Execution Result:
Bonjour Sara,

Je vais être en retard pour notre réunion de demain. Pouvons-nous la reprogrammer?

Cordialement,
Sofia
